# 🧪 Lab 2: Modelling & Model Lifecycle — Predicting Plant Production (GIST Steel Dataset)

---

## 🎯 Learning Outcomes

By completing this lab, you will be able to:

- Prepare and analyse a dataset for modelling, including a schema check before cleaning.  
- Train and evaluate regression models.  
- Apply cross-validation and hyperparameter tuning using scikit-learn.  
- Track experiments and store models using MLflow and/or Optuna.  
- Reflect on the practical aspects of managing the ML lifecycle.

---


In [50]:
import pandas as pd
import numpy as np
import pandera as pa
import mlflow

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV

import joblib
from sklearn.base import clone

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Gabriel DESTAL  Student ID: B00821539
2. Full name: Charles MONTLUC  Student ID: B00821243
3. Full name: Georg Riekhakainen  Student ID: B00818730
4. Full name: Lucas Magret Delord  Student ID: B00819377

**Group / repo name:** `aidams-lab2-destal-montluc-riekhakainen-magretdelord`  
**Submitter (one person):** Gabriel DESTAL

**Repo URL:** https://github.com/GabDestal/aidams-lab2-destal-montluc-riekhakainen-magretdelord

### What to submit

- This notebook (`lab_2.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a **private** GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


## 🧩 1. Data Setup and Exploration

### 🧭 Objective
Understand the dataset structure and the target variable (“plant-level production”).

---

### **Task 1.1 – Load and Inspect Data**
- Load the [GIST Steel dataset](https://globalenergymonitor.org/projects/global-iron-steel-tracker) using [pandas](https://pandas.pydata.org/) (bonus: using [polars](https://pola.rs/).)
- Work at **plant level**: one row per plant. 
- Display basic info (shape, column names, missing values, and data types).
- Identify the target variable (production) and key features (capacity, ...).

Column names differ by release. Inspect `df.columns` and adapt the schema and features to the file you downloaded. [skrub](https://skrub-data.org/)’s `TableReport` is optional here; the modelling use of skrub is in Task 2.1.

In [51]:
# Read the plant, capacity, and production sheets from the workbook.
workbook = "data/Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"
plant = pd.read_excel(workbook, sheet_name="Plant data")
capacity = pd.read_excel(workbook, sheet_name="Plant capacities and status")
production = pd.read_excel(workbook, sheet_name="Plant production")

# Use plant ID as the key and collapse repeated records before joining.
assert plant["GEM plant ID"].is_unique
capacity["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(
    capacity["Nominal crude steel capacity (ttpa)"], errors="coerce"
)
capacity_by_plant = (
    capacity.groupby("GEM plant ID", as_index=False)
    .agg({"Nominal crude steel capacity (ttpa)": "max", "Status": "first"})
)
# Use crude-steel output for one year as the target.
target = "Plant production 2024 (ttpa)"
steel_2024 = production.loc[
    production["Type of production"].eq("Crude steel production (ttpa)"),
    ["GEM plant ID", 2024],
].rename(columns={2024: target}).copy()
steel_2024[target] = pd.to_numeric(steel_2024[target], errors="coerce").astype("float64")

steel_2024 = steel_2024.groupby("GEM plant ID", as_index=False)[target].max()

# Join the sheets while keeping one row per plant.
df = (plant.merge(capacity_by_plant, on="GEM plant ID", how="left")
      .merge(steel_2024, on="GEM plant ID", how="left", validate="one_to_one"))

# Check the finished table and see how much information is missing.
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique plants: {df['GEM plant ID'].nunique():,}")
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes.to_string())
print("\nMissing values (largest counts):\n",
      df.isna().sum().sort_values(ascending=False).head(15).to_string())

key_features = [
    "Nominal crude steel capacity (ttpa)", "Status", "Country/area",
    "Main production equipment", "Plant age",
]
print("\nTarget:", target, "(unreported/'unknown' values are missing)")
print("Key candidate features:", key_features)
print("Plants with reported target:", df[target].notna().sum(), "of", len(df))
display(df[["GEM plant ID", "Plant name (English)", *key_features, target]].head())


Shape: 1,293 rows × 47 columns
Unique plants: 1,293

Columns: ['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source'

,GEM plant ID,Plant name (English),Nominal crude steel capacity (ttpa),Status,Country/area,Main production equipment,Plant age,Plant production 2024 (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,1100.0,operating,Türkiye,EAF,43,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,3000.0,construction,Namibia,EAF,1,NaN
2,P100000120802,Abinsk Electric Steel Works,1600.0,operating,Russia,EAF,12,NaN
3,P100000120020,Abul Khair Steel Sitakunda plant,1400.0,operating,Bangladesh,EAF,11,NaN
4,P100000120620,Acciaierie d'Italia Taranto steel plant,7800.0,announced,Italy,DRI; EAF; BF; BOF,61,2000.0


> 📝 *Critical thinking:*
> Describe any patterns or potential data quality issues you notice. Which variables might strongly influence production?
>
> The workbook separates plant characteristics, capacity/status records, and production by process and year. The merged table uses 2024 crude-steel output as the target; the release represents unreported production with the text `unknown`, which is converted to missing. Capacity records can repeat a plant across equipment/status entries, so the code keeps the maximum reported crude-steel capacity per plant to avoid adding potentially overlapping records. Capacity and operating status should be strong production predictors; country, production equipment, and plant age may capture differences in technology and operating context.

### **Task 1.2 – Schema Check with Pandera**
After Task 1.1, and before any cleaning, write a [Pandera](https://pandera.readthedocs.io/) `DataFrameSchema` for the columns you will use.
- Check dtypes and required columns, including the target.
- Add a few value checks that match what you saw in the file (for example capacity ≥ 0, or status limited to the categories present in the data). Column names vary by GIST release, so build the schema from `df.columns`, not from a guessed list.
- Run `schema.validate(df)` and show what fails.

Use the result in Task 1.3: drop, coerce, or relax a check, and say which.

In [52]:
# Check that the fields needed for this lab exist in this release.
required_columns = [
    "GEM plant ID", target, "Nominal crude steel capacity (ttpa)", "Status"
]
# Stop early if the workbook uses a different schema.
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Required columns missing from this release: {missing_columns}")
columns_to_check = [column for column in df.columns if column in required_columns]
print("Columns being checked:", columns_to_check)

# Require plant IDs and nonnegative production; allow missing capacity and status.
checks = {
    "GEM plant ID": pa.Column(str, nullable=False),
    target: pa.Column(float, nullable=False, checks=pa.Check.ge(0)),
    "Nominal crude steel capacity (ttpa)": pa.Column(
        float, nullable=True, checks=pa.Check.ge(0)
    ),
    "Status": pa.Column(
        str, nullable=True,
        checks=pa.Check.isin(df["Status"].dropna().unique().tolist())
    ),
}
# Build the schema from the columns actually present in the table.
schema_columns = {name: checks[name] for name in columns_to_check}
schema = pa.DataFrameSchema(schema_columns, strict=False)

# Validate the raw data and show any failed checks.
try:
    schema.validate(df, lazy=True)
    print("Schema validation passed.")
except pa.errors.SchemaErrors as error:
    print("Schema validation failed. First failure cases:")
    display(error.failure_cases.head(10))
    print("\nFailure counts by check:")
    display(error.failure_cases["check"].value_counts(dropna=False))


Columns being checked: ['GEM plant ID', 'Nominal crude steel capacity (ttpa)', 'Status', 'Plant production 2024 (ttpa)']
Schema validation failed. First failure cases:


,schema_context,column,check,check_number,failure_case,index
0,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,0
712,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,908
682,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,877
683,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,878
684,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,879
685,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,880
686,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,881
687,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,882
688,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,883
689,Column,Plant production 2024 (ttpa),not_nullable,None,NaN,884



Failure counts by check:


check
not_nullable    1036
Name: count, dtype: int64

> 📝 *Critical thinking:*
> Which checks failed on the raw file, and which of those are real data problems rather than a schema that is stricter than the release?
> What would break in training or in a later deployment if you skipped this check?
>
> The schema requires a plant ID and a nonnegative production target because a training row needs both. Missing production values are expected in this tracker when output was not reported; they are not proof of zero production. Those rows cannot train a supervised model and should be handled in Task 1.3. Capacity and status are allowed to be missing, while negative numeric values and statuses outside the release’s observed categories would indicate a failed check. The status category list is read from this file so the check adapts to the release.

### **Task 1.3 – Data Cleaning**
- Handle missing values appropriately (e.g., imputation, removal).
- Check for outliers or incorrect entries in numerical columns.
- Apply transformations if needed (e.g., log-transform for skewed distributions).


In [53]:
# Keep rows with a known target; the model cannot learn from missing outcomes.
df_clean = df.dropna(subset=[target]).copy()

# Convert numeric-looking fields and leave unrecognized values missing.
numeric_columns = [
    target,
    "Nominal crude steel capacity (ttpa)",
    "Workforce size",
    "Plant age",
]
for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

# Negative production or capacity is not a valid value.
for column in [target, "Nominal crude steel capacity (ttpa)"]:
    if column in df_clean.columns:
        df_clean.loc[df_clean[column] < 0, column] = np.nan

# Review what was removed and inspect possible outliers.
print(f"Rows before cleaning: {len(df):,}")
print(f"Rows with a reported target: {len(df_clean):,}")
print("Missing values in selected numeric columns:")
display(df_clean[[c for c in numeric_columns if c in df_clean.columns]].isna().sum())

for column in [target, "Nominal crude steel capacity (ttpa)"]:
    if column in df_clean.columns:
        q1 = df_clean[column].quantile(0.25)
        q3 = df_clean[column].quantile(0.75)
        iqr = q3 - q1
        outliers = (df_clean[column] < q1 - 1.5 * iqr) | (df_clean[column] > q3 + 1.5 * iqr)
        print(f"Possible IQR outliers in {column}: {outliers.sum()}")

print("\nTarget and capacity summary:")
display(df_clean[[target, "Nominal crude steel capacity (ttpa)"]].describe())
print("Target skew:", round(df_clean[target].skew(), 2))


Rows before cleaning: 1,293
Rows with a reported target: 257
Missing values in selected numeric columns:


Plant production 2024 (ttpa)           0
Nominal crude steel capacity (ttpa)    0
Workforce size                         8
Plant age                              2
dtype: int64

Possible IQR outliers in Plant production 2024 (ttpa): 22
Possible IQR outliers in Nominal crude steel capacity (ttpa): 18

Target and capacity summary:


,Plant production 2024 (ttpa),Nominal crude steel capacity (ttpa)
count,257.000000,257.000000
mean,2319.276265,3212.062257
std,3060.422308,3545.327589
min,1.000000,400.000000
25%,535.000000,950.000000
50%,1012.000000,1720.000000
75%,2890.000000,4102.000000
max,19295.000000,22999.000000


Target skew: 2.72


> 📝 *Critical thinking:*
> Explain your cleaning choices. Why did you treat the missing or skewed data in that way?
>
> We kept only rows with reported 2024 production because the target is needed for supervised training; this leaves 257 plants from 1,293. We converted numeric-looking inputs with `to_numeric`, treating unparseable values as missing rather than zero. Negative production or capacity would be invalid, so the code marks it missing. Missing predictor values remain for later imputation in the model pipeline. The IQR rule flags unusually large production or capacity values, but we retain them because large plants can be genuine. Production is right-skewed, so a log transformation may help a model; we keep this target in original ttpA units for now so results stay interpretable.

### **Task 1.4 – Feature Engineering**
- Create at least two new variables that might improve model performance (e.g., “capacity per worker”, “energy efficiency”).
- Encode categorical variables and standardize numeric ones.
- Bonus: you are free to use external socioeconomic or environmental data sources to enhance your feature set.


In [54]:
# Add two simple features that describe plant scale and age.
capacity_column = "Nominal crude steel capacity (ttpa)"
workforce_column = "Workforce size"

df_clean["Capacity per worker (ttpa)"] = (
    df_clean[capacity_column] / df_clean[workforce_column].replace(0, np.nan)
)

df_clean["Plant age group"] = pd.cut(
    df_clean["Plant age"],
    bins=[0, 20, 50, np.inf],
    labels=["newer", "established", "older"],
    include_lowest=True,
)

# Keep the predictors separate from the production target.
feature_columns = [
    capacity_column,
    workforce_column,
    "Plant age",
    "Capacity per worker (ttpa)",
    "Plant age group",
    "Status",
    "Country/area",
    "Main production equipment",
]
X = df_clean[feature_columns].copy()
y = df_clean[target]

# Split predictors by type so each group gets suitable preprocessing.
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

# Impute and scale numbers; fill and encode categories.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

print("New features: Capacity per worker (ttpa), Plant age group")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("X shape:", X.shape, "| y shape:", y.shape)
display(X.head())


New features: Capacity per worker (ttpa), Plant age group
Numeric features: ['Nominal crude steel capacity (ttpa)', 'Workforce size', 'Plant age', 'Capacity per worker (ttpa)']
Categorical features: ['Plant age group', 'Status', 'Country/area', 'Main production equipment']
X shape: (257, 8) | y shape: (257,)


,Nominal crude steel capacity (ttpa),Workforce size,Plant age,Capacity per worker (ttpa),Plant age group,Status,Country/area,Main production equipment
4,7800.0,11000.0,61.0,0.709091,older,announced,Italy,DRI; EAF; BF; BOF
9,850.0,1319.0,60.0,0.644428,older,operating,South Africa,EAF
10,1200.0,1746.0,56.0,0.687285,older,operating,Spain,EAF
11,1250.0,730.0,39.0,1.712329,established,operating,Peru,EAF; DRI
17,2760.0,3600.0,57.0,0.766667,older,announced,Germany,DRI; EAF; BF; BOF


> 📝 *Critical thinking:*
> Document your new feature(s). What business or operational insight do they represent?
>
> `Capacity per worker (ttpa)` describes installed steelmaking capacity relative to reported workforce size. `Plant age group` captures possible differences in technology and maintenance across newer, established, and older plants. The predictors are separated from `y` so production cannot accidentally become an input. Numeric features are set up for median imputation and standardization; categorical features are set up for imputation and one-hot encoding. The preprocessor is defined here but should be fitted only inside a pipeline on the training split, not on the full dataset.

## 🔍 1.5 Feature Relationships and Correlations

### 🧭 Objective
Before training models, it’s essential to understand how features relate to each other and to the target variable — both linearly and nonlinearly. This helps identify redundant or uninformative predictors and guides model choice.

---

### **Task 1.5.1 – Correlation Matrix (Linear Relationships)**
- Compute a **correlation matrix** (e.g., using `df.corr()`, `seaborn.heatmap`, `skrub`) to examine pairwise linear relationships among numerical features.
- Focus on correlations between each feature and the target (`production`), as well as between features themselves.


In [55]:
# Correlations here use numeric predictors and the target only.
numeric_data = X.select_dtypes(include="number").copy()
numeric_data[target] = y

# Compare straight-line correlations with rank-based correlations.
pearson_corr = numeric_data.corr(method="pearson")
spearman_corr = numeric_data.corr(method="spearman")

print("Pearson correlation matrix:")
display(pearson_corr.round(2))

print("Spearman correlation matrix:")
display(spearman_corr.round(2))

print("\nNumeric features most correlated with production (Pearson):")
display(
    pearson_corr[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
    .to_frame("Correlation with production")
    .round(2)
)

Pearson correlation matrix:


,Nominal crude steel capacity (ttpa),Workforce size,Plant age,Capacity per worker (ttpa),Plant production 2024 (ttpa)
Nominal crude steel capacity (ttpa),1.00,0.58,-0.00,0.03,0.94
Workforce size,0.58,1.00,0.09,-0.19,0.55
Plant age,-0.00,0.09,1.00,-0.14,0.00
Capacity per worker (ttpa),0.03,-0.19,-0.14,1.00,0.03
Plant production 2024 (ttpa),0.94,0.55,0.00,0.03,1.00


Spearman correlation matrix:


,Nominal crude steel capacity (ttpa),Workforce size,Plant age,Capacity per worker (ttpa),Plant production 2024 (ttpa)
Nominal crude steel capacity (ttpa),1.00,0.67,0.08,0.04,0.89
Workforce size,0.67,1.00,0.15,-0.69,0.60
Plant age,0.08,0.15,1.00,-0.11,0.06
Capacity per worker (ttpa),0.04,-0.69,-0.11,1.00,0.04
Plant production 2024 (ttpa),0.89,0.60,0.06,0.04,1.00



Numeric features most correlated with production (Pearson):


,Correlation with production
Nominal crude steel capacity (ttpa),0.94
Workforce size,0.55
Capacity per worker (ttpa),0.03
Plant age,0.00


> 📝 *Critical thinking:*
> Which variables show the strongest correlation with production?
> Do any features appear redundant or highly correlated with each other?
>
> Use the sorted production correlations above to identify the strongest numeric relationships. Pearson captures linear association, while Spearman can show a monotonic relationship when the pattern is not linear. Capacity and workforce may be correlated because larger plants need more workers; `Capacity per worker (ttpa)` is derived from capacity and workforce, so it may also overlap with them. Correlation alone does not show causation, and categorical predictors such as country, status, and equipment are not represented in these matrices.

## 🧮 2. Building Baseline & Linear Models

### 🧭 Objective
Establish a simple baseline, then train and interpret a linear model.

---

### **Task 2.1 – Baseline**
- Compute a simple baseline predictor (mean or median production) with `sklearn.dummy.DummyRegressor`. Report RMSE and MAE on the held-out test set.
- Build a second baseline as a pipeline, using **one** of the following. Fit preprocessing on the training split only (inside the pipeline, not on the full table):
  - **scikit-learn:** a `Pipeline` with preprocessing (`ColumnTransformer`: impute, encode categoricals, scale numerics) and a default regressor.
  - **skrub:** [`tabular_pipeline("regressor")`](https://skrub-data.org/stable/reference/generated/skrub.tabular_pipeline.html), which wires a `TableVectorizer` to an estimator-appropriate preprocessor.
- Compare the pipeline’s RMSE and MAE with the dummy baseline.


In [56]:
# Hold out the same plants for both baseline comparisons.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Baseline 1: predict the average production in the training data for every plant.
# First, compare against predicting the training-set average every time.
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

#Baseline 2: preprocess the features, then fit a default linear regression.
# Then test whether the plant features improve on that simple baseline.
linear_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", LinearRegression()),
])
linear_pipeline.fit(X_train, y_train)
linear_predictions = linear_pipeline.predict(X_test)

# Compare both models with errors on the held-out plants.
baseline_results = pd.DataFrame({
    "Model": ["Mean baseline", "Preprocessing + linear regression"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, dummy_predictions)),
        np.sqrt(mean_squared_error(y_test, linear_predictions)),
    ],
    "MAE": [
        mean_absolute_error(y_test, dummy_predictions),
        mean_absolute_error(y_test, linear_predictions),
    ],
})
display(baseline_results.round(2))

,Model,RMSE,MAE
0,Mean baseline,3905.01,2245.80
1,Preprocessing + linear regression,1277.67,874.56


> 📝 *Critical thinking:*
> Why is it useful to have a baseline model before trying more complex ones?
> Did your pipeline beat the dummy baseline? If the gap is small, what does that say about the features?
>
> The mean baseline gives a useful reference point: it shows the error from predicting the same average output for every plant. Compare the two RMSE and MAE values above. If linear regression improves on the dummy baseline, the available plant features contain predictive information; if not, they may need better features or a model that captures nonlinear relationships.

### **Task 2.2 – Linear Regression**
- Train a multiple linear regression model using the key plant variables.
- Display coefficients and interpret their meaning.
- Evaluate the model on training and test data.


In [57]:
# Fit the linear model with preprocessing inside the pipeline.
multiple_linear_model = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", LinearRegression()),
])
multiple_linear_model.fit(X_train, y_train)

# Compare fit on training plants with performance on held-out plants.
train_predictions = multiple_linear_model.predict(X_train)
test_predictions = multiple_linear_model.predict(X_test)

linear_metrics = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_train, train_predictions)),
        np.sqrt(mean_squared_error(y_test, test_predictions)),
    ],
    "MAE": [
        mean_absolute_error(y_train, train_predictions),
        mean_absolute_error(y_test, test_predictions),
    ],
    "R²": [
        r2_score(y_train, train_predictions),
        r2_score(y_test, test_predictions),
    ],
})
display(linear_metrics.round(2))

# Pair each coefficient with its transformed feature name.
feature_names = multiple_linear_model.named_steps["preprocessing"].get_feature_names_out()
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": multiple_linear_model.named_steps["regression"].coef_,
})
coefficients["Absolute coefficient"] = coefficients["Coefficient"].abs()
display(coefficients.sort_values("Absolute coefficient", ascending=False).head(15).round(2))


,Dataset,RMSE,MAE,R²
0,Train,591.90,346.60,0.96
1,Test,1277.67,874.56,0.89


,Feature,Coefficient,Absolute coefficient
66,categorical__Main production equipment_BF; BOF...,4337.31,4337.31
32,categorical__Country/area_Kazakhstan,-3631.69,3631.69
71,categorical__Main production equipment_BF; BOF...,-3499.71,3499.71
39,categorical__Country/area_Netherlands,2440.99,2440.99
0,numeric__Nominal crude steel capacity (ttpa),2318.65,2318.65
65,categorical__Main production equipment_BF; BOF...,2315.20,2315.20
14,categorical__Country/area_Austria,2009.84,2009.84
84,categorical__Main production equipment_DRI; EA...,-1819.17,1819.17
75,categorical__Main production equipment_BF; EAF,-1761.50,1761.50
73,categorical__Main production equipment_BF; DRI...,-1731.07,1731.07


> 📝 *Critical thinking:*
> Interpret one positive and one negative coefficient. What do they tell you about plant performance drivers?
>
> The training and test metrics show how well the regression fits known plants and generalizes to held-out plants. A large train/test gap can indicate overfitting. Numeric inputs were standardized, so their coefficients describe the expected change in production (ttpa) for a one-standard-deviation increase, holding other features fixed. One-hot encoded category coefficients are differences from the encoder’s reference category. Choose a positive and a negative coefficient from the table and interpret them with these scales in mind; coefficients describe associations, not causal effects.

## 🔁 3. Model Evaluation and Selection

### 🧭 Objective
Use cross-validation to estimate generalization performance and compare multiple model types.

---

### **Task 3.1 – Cross-Validation**
- Apply **K-Fold cross-validation** (e.g., K=5).
- Record the average RMSE, MAE, and R² across folds.


In [58]:
from sklearn.model_selection import KFold, cross_validate

# Shuffle the plants and split them into five repeatable folds.
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate each fold with preprocessing fitted only on its training portion.
cv_scores = cross_validate(
    multiple_linear_model,
    X,
    y,
    cv=cv,
    scoring={
        "rmse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
)

# Show each fold and the average scores.
fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "RMSE": np.sqrt(-cv_scores["test_rmse"]),
    "MAE": -cv_scores["test_mae"],
    "R²": cv_scores["test_r2"],
})
display(fold_results.round(2))
print("Average across folds:")
display(fold_results[["RMSE", "MAE", "R²"]].mean().to_frame("Mean score").round(2))

,Fold,RMSE,MAE,R²
0,1,1277.67,874.56,0.89
1,2,1070.04,730.25,0.86
2,3,1353.67,846.48,0.69
3,4,1103.69,726.19,0.86
4,5,1147.53,705.02,0.84


Average across folds:


,Mean score
RMSE,1190.52
MAE,776.50
R²,0.83


> 📝 *Critical thinking:*
> Summarize your results. How stable is performance across folds? What might this indicate about model variance?
>
> Compare the five fold scores with their averages. Similar scores suggest performance is reasonably stable across different train/validation splits; a wide spread suggests results depend more on which plants are held out. With this relatively small set of plants with reported production, some variation across folds is expected.

### **Task 3.2 – Model Comparison**
Train and compare at least **three models**, on the same folds and metrics as Task 3.1:
- Linear Regression
- Ridge Regression (regularized linear)
- Random Forest Regressor

Record cross-validation performance for each model.

**Bonus — tabular foundation model:** add [TabFM](https://github.com/google-research/tabfm) v1.0 (Google Research). Install the PyTorch extra from that repo, then:

```python
from tabfm import TabFMRegressor
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

reg = TabFMRegressor(model=tabfm_v1_0_0.load(model_type="regression"))
```

`fit` stores the training rows as context; `predict` is one forward pass, with no hyperparameter search. The default context is 100 rows (`max_num_rows`). A single held-out split is enough. Pretrained weights are non-commercial and not for production. In the results table, say whether it beats your best classical model, and which model you would actually deploy. Python 3.11 or newer is required.


In [59]:
# Compare two linear models with a tree-based model.
models = {
    "Linear regression": LinearRegression(),
    "Ridge regression": Ridge(),
    "Random forest": RandomForestRegressor(random_state=42),
}

# Use the same folds and metrics for every model.
comparison_rows = []
for model_name, estimator in models.items():
    model_pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("regression", estimator),
    ])
    scores = cross_validate(
        model_pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "rmse": "neg_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
    )
    comparison_rows.append({
        "Model": model_name,
        "Mean RMSE": np.sqrt(-scores["test_rmse"]).mean(),
        "RMSE std": np.sqrt(-scores["test_rmse"]).std(),
        "Mean MAE": -scores["test_mae"].mean(),
        "MAE std": (-scores["test_mae"]).std(),
        "Mean R²": scores["test_r2"].mean(),
        "R² std": scores["test_r2"].std(),
    })

# Rank models by average cross-validation RMSE.
comparison_results = pd.DataFrame(comparison_rows).sort_values("Mean RMSE")
display(comparison_results.round(2))

best_model_name = comparison_results.iloc[0]["Model"]
best_estimator = models[best_model_name]
print("Best model by mean CV RMSE:", best_model_name)


,Model,Mean RMSE,RMSE std,Mean MAE,MAE std,Mean R²,R² std
1,Ridge regression,1048.79,173.83,661.16,72.38,0.86,0.08
0,Linear regression,1190.52,107.81,776.50,69.70,0.83,0.07
2,Random forest,1220.56,254.88,654.33,139.18,0.81,0.10


Best model by mean CV RMSE: Ridge regression


> 📝 *Critical thinking:*
> Create a small results table. Which model performs best? Why might that be the case given the dataset’s characteristics?
>
> Compare mean RMSE, MAE, and R², and use the standard deviations to judge how consistent each model is across folds. The model with the lowest mean RMSE is selected for Task 3.3. A tree model can capture nonlinear patterns and interactions, while the linear models are simpler to interpret. Results may still be limited by the small number of plants with reported production and by missing predictors.

### **Task 3.3 – Hyperparameter Optimization**
- Use **RandomizedSearchCV** or **GridSearchCV** to tune the top model (e.g., Random Forest).
- Report the best parameters and corresponding validation score.


In [60]:
# Choose search ranges for the best model from Task 3.2.
if best_model_name == "Random forest":
    tuning_name = "Random forest"
    tuning_estimator = RandomForestRegressor(random_state=42)
    parameter_choices = {
        "regression__n_estimators": [100, 200, 400],
        "regression__max_depth": [None, 5, 10],
        "regression__min_samples_leaf": [1, 2, 4],
    }
elif best_model_name == "Ridge regression":
    tuning_name = "Ridge regression"
    tuning_estimator = Ridge()
    parameter_choices = {"regression__alpha": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]}
else:
    tuning_name = "Ridge regression (linear model alternative)"
    tuning_estimator = Ridge()
    parameter_choices = {"regression__alpha": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]}

# Keep preprocessing inside the search so each fold is handled separately.
tuning_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("regression", tuning_estimator),
])
search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=parameter_choices,
    n_iter=10,
    scoring={"rmse": "neg_mean_squared_error", "r2": "r2"},
    refit="rmse",
    cv=cv,
    random_state=42,
)
# Try ten parameter combinations using five-fold validation.
search.fit(X, y)

# Report the best settings and compare them with the untuned score.
best_cv_rmse = np.sqrt(-search.best_score_)
best_cv_r2 = search.cv_results_["mean_test_r2"][search.best_index_]
print("Model tuned:", tuning_name)
print("Best parameters:", search.best_params_)
print("Best cross-validation RMSE:", round(best_cv_rmse, 2))
print("Best cross-validation R²:", round(best_cv_r2, 2))

untuned_name = "Ridge regression" if best_model_name == "Linear regression" else best_model_name
untuned_rmse = comparison_results.loc[
    comparison_results["Model"].eq(untuned_name), "Mean RMSE"
].iloc[0]
print("Untuned cross-validation RMSE:", round(untuned_rmse, 2))

Model tuned: Ridge regression
Best parameters: {'regression__alpha': 1}
Best cross-validation RMSE: 1063.1
Best cross-validation R²: 0.86
Untuned cross-validation RMSE: 1048.79


> 📝 *Critical thinking:*
> Discuss the role of hyperparameter tuning. How did tuning change your model’s performance compared to default settings?
>
> Compare the best cross-validation RMSE from the randomized search with the untuned score printed above. A lower score means the searched settings improved validation performance; a small change may mean the default model was already reasonable or that the features limit performance. When linear regression was the best model in Task 3.2, Ridge was tuned because ordinary linear regression has no meaningful hyperparameter to optimize.

## ⚙️ 4. Model Lifecycle: Tracking, Saving, and Loading

### 🧭 Objective
Apply tools that support reproducible ML experiments.

---

### **Task 4.1 – Experiment Tracking with MLflow**
- Use MLflow to log parameters (model type, hyperparameters), metrics (RMSE, R²), and artifacts (a plot, the feature list, or your Pandera schema).
- Run and record at least two experiments (for example the Task 2.1 baseline and your best model).


In [61]:
# Store both experiments in one named MLflow experiment.
experiment_name = "GIST Steel Plant Production"
mlflow.set_experiment(experiment_name)

# Measure the mean baseline on the same cross-validation folds.
dummy_cv = cross_validate(
    DummyRegressor(strategy="mean"),
    X,
    y,
    cv=cv,
    scoring={"rmse": "neg_mean_squared_error", "r2": "r2"},
)
dummy_cv_rmse = np.sqrt(-dummy_cv["test_rmse"]).mean()
dummy_cv_r2 = dummy_cv["test_r2"].mean()

# Record parameters, scores, and the feature list for each run.
logged_runs = []

# Log the simple baseline first.
with mlflow.start_run(run_name="Mean baseline") as run:
    mlflow.log_param("model_type", "DummyRegressor")
    mlflow.log_param("strategy", "mean")
    mlflow.log_metric("cv_rmse", dummy_cv_rmse)
    mlflow.log_metric("cv_r2", dummy_cv_r2)
    mlflow.log_text("\n".join(feature_columns), "feature_list.txt")
    logged_runs.append({"Run": "Mean baseline", "Run ID": run.info.run_id,
                        "CV RMSE": dummy_cv_rmse, "CV R²": dummy_cv_r2})

# Log the tuned model and its selected parameters.
with mlflow.start_run(run_name="Tuned model") as run:
    mlflow.log_param("model_type", tuning_name)
    for parameter, value in search.best_params_.items():
        mlflow.log_param(parameter, str(value))
    mlflow.log_metric("cv_rmse", best_cv_rmse)
    mlflow.log_metric("cv_r2", best_cv_r2)
    mlflow.log_text("\n".join(feature_columns), "feature_list.txt")
    logged_runs.append({"Run": "Tuned model", "Run ID": run.info.run_id,
                        "CV RMSE": best_cv_rmse, "CV R²": best_cv_r2})

display(pd.DataFrame(logged_runs).round(3))
print("MLflow experiment:", experiment_name)

,Run,Run ID,CV RMSE,CV R²
0,Mean baseline,93a067f105864e5d8fe15b10c744f974,3016.553,-0.007
1,Tuned model,df1e4cc4f1444889ad259774fabf6c37,1063.101,0.861


MLflow experiment: GIST Steel Plant Production


> 📝 *Critical thinking:*
> Describe how MLflow helps manage your experiments. What advantages does it give compared to manual tracking?
>
> MLflow records each run’s model settings, cross-validation metrics, and feature list in one place. This makes it easier to compare experiments and reproduce which settings produced a score than keeping results in separate notes or tables.

### **Task 4.2 – Hyperparameter Optimization with Optuna**
- Define an Optuna study to optimize one classical model (Ridge or Random Forest). Skip TabFM here.
- Use a fixed budget (about 30 trials) and the same cross-validation metric as Task 3.
- Record the number of trials, the best parameters, and the score next to the untuned score from Task 3.2.
- Send the trials to the same store as Task 4.1 with `optuna.integration.MLflowCallback`.


In [62]:
# Load Optuna and its MLflow callback for recording the search trials.
import optuna
try:
    from optuna.integration import MLflowCallback
except ImportError:
    try:
        from optuna_integration.mlflow import MLflowCallback
    except ImportError as error:
        raise ImportError(
            'Install the MLflow callback with %pip install "optuna-integration[mlflow]"'
        ) from error
# Each trial builds a forest and scores it with five-fold RMSE.
def objective(trial):
    # Optuna proposes a set of Random Forest settings for each trial.
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 300, step=100),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        random_state=42,
    )
    model_pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("regression", model),
    ])

    fold_mse = cross_validate(
        model_pipeline,
        X,
        y,
        cv=cv,
        scoring="neg_mean_squared_error",
    )["test_score"]
    return np.sqrt(-fold_mse).mean()

# Send trial runs to the experiment created in Task 4.1.
experiment = mlflow.get_experiment_by_name(experiment_name)
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="cv_rmse",
    create_experiment=False,
    mlflow_kwargs={"experiment_id": experiment.experiment_id},
)

# Run the 30-trial search and keep the best result.
study = optuna.create_study(direction="minimize", study_name="Random Forest tuning")
study.optimize(objective, n_trials=30, callbacks=[mlflow_callback])

# Compare the tuned score with the default forest from Task 3.2.
untuned_rf_rmse = comparison_results.loc[
    comparison_results["Model"].eq("Random forest"), "Mean RMSE"
].iloc[0]
optuna_results = pd.DataFrame({
    "Result": ["Untuned Random Forest (Task 3.2)", "Optuna tuned Random Forest"],
    "CV RMSE": [untuned_rf_rmse, study.best_value],
})

print("Trials completed:", len(study.trials))
print("Best parameters:", study.best_params)
display(optuna_results.round(2))
print("Optuna trials were logged to MLflow experiment:", experiment_name)

C:\Users\gdest\AppData\Local\Temp\ipykernel_28716\542634719.py:37: FutureWarning: MLflowCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  mlflow_callback = MLflowCallback(
[I 2026-09-25 11:08:14,463] A new study created in memory with name: Random Forest tuning
[I 2026-09-25 11:08:15,803] Trial 0 finished with value: 1304.2631846778677 and parameters: {'n_estimators': 100, 'max_depth': 13, 'min_samples_leaf': 5}. Best is trial 0 with value: 1304.2631846778677.
[I 2026-09-25 11:08:19,517] Trial 1 finished with value: 1303.1469378557485 and parameters: {'n_estimators': 300, 'max_depth': 7, 'min_samples_leaf': 5}. Best is trial 1 with value: 1303.1469378557485.
[I 2026-09-25 11:08:24,327] Trial 2 finished with value: 1249.0545580054888 and parameters: {'n_estimators': 300, 'max_depth': 17, 'min_samples_leaf': 2}. Best is trial 2 with value: 1249.0545580054888.
[I 2026-09-25 11:08:28,080] Trial 3 fini

Trials completed: 30
Best parameters: {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 1}


,Result,CV RMSE
0,Untuned Random Forest (Task 3.2),1220.56
1,Optuna tuned Random Forest,1215.00


Optuna trials were logged to MLflow experiment: GIST Steel Plant Production


> 📝 *Critical thinking:*
> Describe how MLflow helps manage your experiments. What advantages does it give compared to manual tracking?
>
> Optuna proposes hyperparameter settings, evaluates each one with the same five-fold RMSE, and uses earlier trial results to guide later proposals. This can explore a broad parameter space more efficiently than checking every combination in a grid. Compare the tuned score with the untuned Random Forest score from Task 3.2; with only 30 trials, the best result is not guaranteed to be the global optimum.

### **Task 4.3 – Model Storage**
- Save the best **pipeline** (preprocessing and model together), with `joblib` or `mlflow.sklearn.log_model`. Saving the estimator alone drops the encoder and imputer.
- Load the saved pipeline and re-evaluate it on the test set. The score should match the in-memory pipeline.


In [63]:
# Refit the selected pipeline on training data, then save the whole pipeline.
best_pipeline = clone(search.best_estimator_)
best_pipeline.fit(X_train, y_train)

model_path = "best_plant_production_pipeline.joblib"
joblib.dump(best_pipeline, model_path)

# Load the saved pipeline and compare it with the in-memory version.
# Reload it and check that its held-out predictions match.
loaded_pipeline = joblib.load(model_path)
memory_predictions = best_pipeline.predict(X_test)
loaded_predictions = loaded_pipeline.predict(X_test)

print("Saved pipeline:", model_path)
print("Predictions match:", np.allclose(memory_predictions, loaded_predictions))

# Confirm both versions give the same test errors.
saved_model_metrics = pd.DataFrame({
    "Version": ["In memory", "Loaded from file"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, memory_predictions)),
        np.sqrt(mean_squared_error(y_test, loaded_predictions)),
    ],
    "MAE": [
        mean_absolute_error(y_test, memory_predictions),
        mean_absolute_error(y_test, loaded_predictions),
    ],
})
display(saved_model_metrics.round(2))


Saved pipeline: best_plant_production_pipeline.joblib
Predictions match: True


,Version,RMSE,MAE
0,In memory,1102.13,710.65
1,Loaded from file,1102.13,710.65


> 📝 *Critical thinking:*
> Why is it important to store both model parameters and metadata? How would you ensure version control of models in a production setting?
>
> The saved file contains both preprocessing and the fitted estimator, so new rows receive the same imputation, scaling, and category encoding used during training. Keeping the pipeline and model metadata together makes it easier to reproduce predictions and identify which version is deployed. In production, I would keep versioned model artifacts with the code, data schema, and training metrics.

## 🚀 5. Deployment & Monitoring (Conceptual)

### 🧭 Objective
Reflect on how models transition from training to production and stay reliable over time.

---

### **Task 5.1 – Deployment Planning**

> 📝 *Critical thinking:*
> Describe how you would deploy the saved pipeline from Task 4.3 (REST API or batch job). Incoming rows should pass your Pandera schema before predict, so serving applies the same columns and checks as training. Which metrics would you monitor: error once actual production arrives, and the share of rows rejected by the schema? If you tried TabFM, would you deploy it or the classical pipeline? The pretrained weights are non-commercial and not for production use.

We would deploy `best_plant_production_pipeline.joblib` as a batch job that loads each new plant row, checks it with a Pandera **feature-only** schema, and then passes the validated feature columns to `predict`. The serving schema should require the same predictors and value rules used at training, but omit the production target because it is not available when making a prediction. The saved pipeline applies the fitted imputation, scaling, and category encoding automatically. We would record each prediction with a plant ID and timestamp.

We would monitor the share of incoming rows rejected by the schema, along with missing-value rates and changes in key inputs such as capacity or plant status. When actual production is reported, we would calculate MAE and RMSE and compare them with the validation results and a simple mean baseline. We did not try TabFM, so we would deploy the classical scikit-learn pipeline saved in Task 4.3; it is the model evaluated here and can be packaged with its preprocessing steps.

### **Task 5.2 – Detecting Model Drift**

> 📝 *Critical thinking:*
> What signs might indicate your model needs retraining? Give one example of data drift and one of concept drift relevant to steel plant production. Name one column from your Pandera schema you would watch first, and what shift in that column would count as data drift.

 We would consider retraining if MAE or RMSE rises over time, if the share of rows rejected by the schema increases, or if important input distributions shift and stay shifted. For example, **data drift** could occur if the mix of plant capacities or operating statuses changes because many new plants enter the tracker. **Concept drift** could occur if energy prices, policy, or demand change so that plants with similar capacity and status produce at different levels than they did during training.

We would watch `Nominal crude steel capacity (ttpa)`, which is included in the Pandera checks. A sustained change in its median or in the share of plants above the training set’s 95th percentile would count as data drift, even if all values still pass the schema’s nonnegative check. We would compare those measures with the training data and investigate persistent changes before retraining.

## 💬 6. Feedback

1. The most challenging step was preparing the tracker for modeling. Plant characteristics, capacity, and production were on separate sheets, production was not reported for many plants, and some plant IDs appeared more than once in the production records. Checking keys and defining one production target per plant was important before fitting models.

2. With more data, we would add plant-level production by year, operating hours or utilization, product mix, equipment age, energy use and prices, and planned outages. These could help explain why production differs among plants with similar nominal capacity.

3. We would explain the model in terms of estimated annual production in thousand tonnes, show its typical test error, and describe which plant characteristics are associated with higher or lower predictions. We would also explain that the model is a planning aid, that estimates have uncertainty, and that these associations do not prove that changing a feature will cause production to change.